In [ ]:
# Colab 請選 2026.07；套件裝入獨立環境，避免覆寫已載入的 NumPy。
from pathlib import Path
import subprocess, sys, tempfile
assert sys.version_info[:3] == (3, 12, 13), "請使用 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch13-book-env")
    if not (env / "bin/python").exists():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["numpy==1.26.4", "pandas==3.0.6", "matplotlib==3.11.2", "rdkit==2026.9.1"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "13-02-negative-results.py"


# 低產率、零值與缺值，該怎麼記錄？

使用 Perera 等人（2018，DOI: 10.1126/science.aap9112）的真實反應紀錄；來源為 Open Reaction Database（Kearnes 等，2021，DOI: 10.1021/jacs.1c09820）。固定 commit 2a6dff66c02f5d1c9b1589691f845b941c981a7a。

資料快照以 gzip/base64 內嵌，執行前檢查 SHA-256；保留全部 5,760 筆，僅投影必要欄位，未填補產率或合成資料，移除提交者聯絡欄位。資料及其投影採 CC BY-SA 4.0：https://creativecommons.org/licenses/by-sa/4.0/ 。程式為本書新寫。

[原始資料](https://github.com/open-reaction-database/ord-data/blob/2a6dff66c02f5d1c9b1589691f845b941c981a7a/data/68/ord_dataset-68cb8b4b2b384e3d85b5b1efae58b203.pb.gz)。資料不需網路或額外 ORD 套件，首次在 Colab 安裝套件需要網路。圖形英文字與程式欄位對應，判讀見書稿。


In [ ]:
%%writefile $script
import base64, gzip, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
SNAPSHOT = '...'
SNAPSHOT_SHA = '02bacbe9789b2f818e82f75b2667396f236842e80bb48545f95ddaca367bb680'
blob = base64.b64decode(SNAPSHOT)
assert hashlib.sha256(blob).hexdigest() == SNAPSHOT_SHA
payload = json.loads(gzip.decompress(blob))
records = payload['records']
assert len(records) == 5760
figdir = Path('figures'); figdir.mkdir(exist_ok=True)
rx, cp, ms = [], [], []
for r in records:
    rid = r['reaction_id']; cond = r['conditions']
    rx.append({'reaction_id': rid, 'temperature_value': cond['temperature']['setpoint']['value'],
               'temperature_unit': cond['temperature']['setpoint']['units'],
               'pressure_value': cond['pressure']['setpoint']['value'],
               'pressure_unit': cond['pressure']['setpoint']['units'], 'doi': r['doi'],
               'has_notes': r['has_notes'], 'has_observations': r['has_observations']})
    for stream, inp in r['inputs'].items():
        for ci, c in enumerate(inp['components']):
            smiles = next((i['value'] for i in c['identifiers'] if i['type'] == 'SMILES'), None)
            a = c.get('amount', {}); m = a.get('moles', {}); v = a.get('volume', {})
            cp.append({'reaction_id': rid, 'stream': stream, 'component_index': ci,
                       'smiles': smiles, 'role': c.get('reaction_role'),
                       'moles_value': m.get('value'), 'moles_unit': m.get('units'),
                       'volume_value': v.get('value'), 'volume_unit': v.get('units'),
                       'volume_includes_solutes': a.get('volume_includes_solutes')})
    for oi, outcome in enumerate(r['outcomes']):
        for pi, product in enumerate(outcome['products']):
            for mi, m in enumerate(product['measurements']):
                a = outcome['analyses'][m['analysis_key']]
                ms.append({'reaction_id': rid, 'outcome_index': oi, 'product_index': pi,
                           'measurement_index': mi, 'measurement_type': m['type'],
                           'yield_pct': m.get('percentage', {}).get('value'),
                           'analysis_key': m['analysis_key'], 'analysis_type': a['type'],
                           'analysis_details': a.get('details'),
                           'uses_internal_standard': m.get('uses_internal_standard'),
                           'uses_authentic_standard': m.get('uses_authentic_standard'),
                           'is_normalized': m.get('is_normalized'),
                           'reaction_time_value': outcome['reaction_time']['value'],
                           'reaction_time_unit': outcome['reaction_time']['units']})
reactions = pd.DataFrame(rx); components = pd.DataFrame(cp); measurements = pd.DataFrame(ms)
assert reactions.reaction_id.is_unique
assert measurements.reaction_id.is_unique  # This dataset only, not a general ORD rule.
assert not components.duplicated(['reaction_id', 'stream', 'component_index']).any()
assert set(components.reaction_id) == set(reactions.reaction_id)
assert set(measurements.reaction_id) == set(reactions.reaction_id)
y = measurements['yield_pct']
assert y.notna().all() and y.between(0, 100).all()
# This dataset has explicit control streams: stream missing remains unknown.
def stream_state(r, name, role):
    inp = r['inputs'].get(name)
    if inp is None:
        return 'not_recorded'
    return 'recorded_component' if any(c.get('reaction_role') == role for c in inp['components']) else 'solvent_only'
ligand_states = pd.Series([stream_state(r, 'Ligand in Solvent', 'CATALYST') for r in records], name='ligand_stream_state')
print('dataset', payload['dataset_id'], 'source_commit', payload['source_commit'])
print('tables', len(reactions), len(components), len(measurements))

result_status = pd.Series(np.where(y.eq(0), 'reported_zero', 'reported_positive'), index=y.index)
kept = y[y > 0]
print('reported_zero', int(y.eq(0).sum()), 'positive', int(y.gt(0).sum()), 'missing', int(y.isna().sum()))
print('means', float(y.mean()), float(kept.mean()), 'medians', float(y.median()), float(kept.median()))
print('ligand_states', ligand_states.value_counts().to_dict())
assert ligand_states.value_counts()['solvent_only'] == 480
assert not ligand_states.eq('not_recorded').any()
# Mark this as a filtering rule, not a physical detection limit.
print('below_5_percent', int(y.lt(5).sum()))
fig, axes = plt.subplots(1,2,figsize=(10,4.4))
bins=np.arange(0,101,10)
axes[0].hist(y,bins=bins,weights=np.ones(len(y))/len(y)*100,alpha=.65,label=f'All (n={len(y):,})',color='#315c78')
axes[0].hist(kept,bins=bins,weights=np.ones(len(kept))/len(kept)*100,histtype='step',linewidth=2,label=f'Zeros removed (n={len(kept):,})',color='#c57945')
axes[0].set(xlabel='ORD reported yield (%)',ylabel='Share within each group (%)',title='Removing zeros changes the sample');axes[0].legend(fontsize=8)
counts=ligand_states.value_counts().reindex(['recorded_component','solvent_only','not_recorded'],fill_value=0)
axes[1].bar(['Component\nrecorded','Solvent\nonly','Stream not\nrecorded'],counts,color=['#315c78','#5a8b78','#bbb'])
for i,v in enumerate(counts):axes[1].text(i,v+80,str(v),ha='center')
axes[1].set(ylim=(0,6100),ylabel='Reactions',title='Read the control stream before filling blanks')
fig.tight_layout();fig.savefig(figdir/'13-02-negative-results.png',dpi=180);plt.close(fig)


In [ ]:
subprocess.run([python, str(script)], cwd=work.name, check=True)
from IPython.display import display, Image, FileLink
import shutil
out = Path("figures"); out.mkdir(exist_ok=True)
figure = Path(work.name) / "figures/13-02-negative-results.png"
shutil.copy2(figure, out / figure.name)
display(Image(filename=str(figure)))
